In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session

# Snowflakeセッション接続とコンテキスト設定
session = get_active_session()
session.sql("USE DATABASE KAGGLE_SUBSCRIPTION").collect()
session.sql("USE SCHEMA DBT_PJT").collect()
table_name = "MART_MONTHLY__ACCOUNT_AGG"

# 1. データベースから生のPandas DataFrameを取得
df_pd = session.sql(f"SELECT * FROM {table_name}").to_pandas()

In [ ]:
# ステータス別・月次でのユニーク顧客数を集計（縦長）
df_bar_raw = (
    df_pd.groupby(["CAL_MONTH", "PAYMENT_STATUS"])
    .agg({"ACCOUNT_ID": "nunique"})
    .reset_index()
)
df_bar_raw.head()

In [ ]:
# 💡【重要】ピボットをかけて、Streamlitが100%誤認しない「ワイドフォーマット」へ成形
df_bar_wide = df_bar_raw.pivot(
    index="CAL_MONTH", 
    columns="PAYMENT_STATUS", 
    values="ACCOUNT_ID"
).fillna(0)

In [ ]:
base_df = (
    df_pd.groupby(['CAL_MONTH']).agg(
    {
        "MRR_AMOUNT": "sum",
        "ACT_MONTH_USD": "sum"
    })
    .reset_index()
    .rename(columns = {"MRR_AMOUNT": "Expected", "ACT_MONTH_USD":"Actual"})
)

tmp_df = (
    df_pd[df_pd["PAYMENT_STATUS"]!="Stop Paying"]
    .groupby(["CAL_MONTH"])[["ACCOUNT_ID"]]
    .nunique()
    .reset_index()
    .rename(columns={"ACCOUNT_ID":"ActiveUU"})
)

In [ ]:
result = pd.merge(base_df, tmp_df, on=["CAL_MONTH"], how="left")
result.index = result["CAL_MONTH"]

result.head()